In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'e:\\ml projects\\fraud-transaction-detection'

In [4]:
from pathlib import Path
from dataclasses import dataclass 

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path 
    train_data_path: Path
    test_data_path: Path
    model_name: str
    subsample: float
    reg_lambda: int
    reg_alpha: float
    n_estimators: int
    min_child_weight: int
    max_depth: int
    learning_rate: float
    colsample_bytree: float
    target_column: str
    

In [5]:
from src.transaction_fraud_detection.constants import * 
from src.transaction_fraud_detection.utils.common import read_yaml,create_directories

In [6]:
class ConfigManager:
    def __init__(
        self,
        config_file=CONFIG_FILE_PATH, 
        schema_file=SCHEMA_FILE_PATH,
        params_file=PARAM_FILE_PATH):

        self.config=read_yaml(config_file)
        self.schema=read_yaml(schema_file)
        self.params=read_yaml(params_file)
        
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainerConfig:
        config=self.config.model_trainer
        schema=self.schema.TARGET_COLUMN
        params=self.params.XGBClassifier

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            subsample=params.subsample,
            reg_lambda=params.reg_lambda,
            reg_alpha=params.reg_alpha,
            n_estimators=params.n_estimators,
            min_child_weight=params.min_child_weight,
            max_depth=params.max_depth,
            learning_rate=params.learning_rate,
            colsample_bytree=params.colsample_bytree,
            target_column=list(schema.keys())[0])
        return model_trainer_config

In [7]:
from xgboost import XGBClassifier 
from src.transaction_fraud_detection.logging import logger
import pandas as pd 
import joblib

[2025-08-17 14:13:47,472 : INFO : utils : NumExpr defaulting to 12 threads.]


In [8]:
class ModelTraining:
    def __init__(self,config:ModelTrainerConfig):
        self.config=config

    def model_training(self):
            train_data=pd.read_csv(self.config.train_data_path)
            test_data=pd.read_csv(self.config.test_data_path)

            train_x=train_data.drop([self.config.target_column],axis=1)
            train_y=train_data[[self.config.target_column]]
            test_x=test_data.drop([self.config.target_column],axis=1)
            test_y=test_data[[self.config.target_column]]

            xgb=XGBClassifier( subsample=0.8,reg_lambda=1,reg_alpha=0.1,n_estimators=150,min_child_weight=1,max_depth=30,learning_rate=0.02,colsample_bytree=0.6)

            xgb.fit(train_x,train_y)

            joblib.dump(xgb,os.path.join(self.config.root_dir,self.config.model_name))

        

In [9]:
try: 
    config=ConfigManager()
    model_training_config=config.get_model_trainer_config()
    model_training=ModelTraining(config=model_training_config)
    model_training.model_training()
except Exception as e:
    raise e

[2025-08-17 14:13:48,299 : INFO : common : yaml_fileconfig\config.yaml loaded sucessfully]
[2025-08-17 14:13:48,301 : INFO : common : yaml_fileschema.yaml loaded sucessfully]
[2025-08-17 14:13:48,302 : INFO : common : yaml_fileparams.yaml loaded sucessfully]
[2025-08-17 14:13:48,303 : INFO : common : create directory at artifacts]
[2025-08-17 14:13:48,304 : INFO : common : create directory at artifacts/model_trainer]
